# 04 — Avaliação e validação

A última etapa junta as três condições: modelo original, ablação da direção e adição da direção. Além das taxas de recusa, calculamos a variação de entropia cruzada em instruções inofensivas para verificar se a intervenção apenas destruiu a linguagem.

A hipótese causal exige um padrão conjunto: a ablação reduz a recusa nociva, a adição aumenta falsas recusas e o controle aleatório não reproduz o efeito principal.

In [ ]:
# Mantemos a mesma direção escolhida e calculamos as métricas de preservação na amostra de validação.
from pathlib import Path
import json

from refusal_direction_replication.pipeline import run_evaluation_stage

cwd = Path.cwd()
roots = (cwd, cwd.parent, cwd / "refusal_direction", Path("/content/refusal_direction"))
ROOT = next((path for path in roots if (path / "configs" / "gemma_2b.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Não foi possível localizar a raiz de refusal_direction.")
config = json.loads((ROOT / "configs" / "gemma_2b.json").read_text(encoding="utf-8"))
config

## Como evitar uma conclusão fácil demais

A taxa de recusa sozinha não informa se o modelo continuou linguisticamente funcional. Por isso comparamos a entropia cruzada por exemplo no conjunto inofensivo. Também mantemos o controle aleatório de mesma norma, que separa o efeito da direção escolhida do efeito genérico de perturbar o residual stream.

Os resultados são uma replicação focada do mecanismo, não uma prova de que uma única direção contém todo o alinhamento do modelo.

In [ ]:
# Calculamos a entropia cruzada nas quatro condições e consolidamos o artefato final.
evaluation = run_evaluation_stage(ROOT)
evaluation_results = evaluation["results"]
evaluation_results

In [ ]:
# Tornamos explícitos os três testes causais e os deltas individuais de entropia cruzada.
causal_checks = {
    "ablation_reduces_harmful_refusal": evaluation_results["ablation"]["harmful_refusal_rate"] < evaluation_results["baseline"]["harmful_refusal_rate"],
    "addition_increases_false_refusal": evaluation_results["addition"]["harmless_refusal_rate"] > evaluation_results["baseline"]["harmless_refusal_rate"],
    "random_control_is_weaker_than_ablation": evaluation_results["random_control"]["harmful_refusal_rate"] > evaluation_results["ablation"]["harmful_refusal_rate"],
}
per_example_ce_delta = [
    changed - baseline
    for baseline, changed in zip(
        evaluation_results["per_example_cross_entropy"]["baseline"],
        evaluation_results["per_example_cross_entropy"]["ablation"],
    )
]
causal_checks, per_example_ce_delta

## Conclusão e limites

Se os três testes causais forem verdadeiros e o delta de entropia cruzada da ablação permanecer abaixo da tolerância configurada, temos uma replicação focada do principal resultado: uma direção de baixa dimensão medeia parte importante da recusa do Gemma 2B IT.

Isso não demonstra que a direção seja todo o alinhamento, nem que funcione fora das distribuições avaliadas. Não reproduzimos a varredura de 13 modelos, ortogonalização de pesos, LoRA ou otimização de sufixos adversariais. O artefato `results/final_evaluation.json` registra os valores observados, a revisão do modelo e o ambiente para que essa conclusão permaneça verificável.